##### Install requirements

In [2]:
import pandas as pd
import requests 


##### Loading the CSV file

In [3]:
takeoffs2016 = pd.read_csv("data/2016.csv")


In [4]:
takeoffs2017 = pd.read_csv("data/2017.csv")

In [5]:
takeoffs2018 = pd.read_csv("data/2018.csv")

#### Create Launch Sites

In [6]:
print(takeoffs2016.head())

      FL_DATE OP_CARRIER  OP_CARRIER_FL_NUM ORIGIN DEST  CRS_DEP_TIME  \
0  2016-01-01         DL               1248    DTW  LAX          1935   
1  2016-01-01         DL               1251    ATL  GRR          2125   
2  2016-01-01         DL               1254    LAX  ATL          2255   
3  2016-01-01         DL               1255    SLC  ATL          1656   
4  2016-01-01         DL               1256    BZN  MSP           900   

   DEP_TIME  DEP_DELAY  TAXI_OUT  WHEELS_OFF  ...  CRS_ELAPSED_TIME  \
0    1935.0        0.0      23.0      1958.0  ...             309.0   
1    2130.0        5.0      13.0      2143.0  ...             116.0   
2    2256.0        1.0      19.0      2315.0  ...             245.0   
3    1700.0        4.0      12.0      1712.0  ...             213.0   
4    1012.0       72.0      63.0      1115.0  ...             136.0   

   ACTUAL_ELAPSED_TIME  AIR_TIME  DISTANCE  CARRIER_DELAY  WEATHER_DELAY  \
0                285.0     249.0    1979.0            NaN 

In [7]:
takeoffs2016['ORIGIN'].unique() 

<StringArray>
['DTW', 'ATL', 'LAX', 'SLC', 'BZN', 'BNA', 'JAX', 'MSP', 'MDT', 'SAV',
 ...
 'GCK', 'GST', 'AKN', 'DLG', 'ABI', 'GRI', 'EFD', 'SPN', 'PGD', 'ENV']
Length: 313, dtype: str

In [8]:
AIRPORTS = {
    'Atlanta ATL': (36.63,  84.43),
    'Los Angeles LAX': (28.50,  -80.57),
    'Pittsburgh PIT': (34.74, -120.57),
    'Seattle SEA': (45.92,   63.34),
    'Dallas Fort Worth DFW': (5.24,  -52.77),
    'Denver DEN': (13.73,   80.23),
}

print(f'Airports being used: {len(AIRPORTS)}')
for name, (lat, lon) in AIRPORTS.items():
    print(f'  {name}: ({lat}, {lon})')

Airports being used: 6
  Atlanta ATL: (36.63, 84.43)
  Los Angeles LAX: (28.5, -80.57)
  Pittsburgh PIT: (34.74, -120.57)
  Seattle SEA: (45.92, 63.34)
  Dallas Fort Worth DFW: (5.24, -52.77)
  Denver DEN: (13.73, 80.23)


In [9]:
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 35.32,
    "longitude": 75.38,
    "start_date":"2016-01-01",
    "end_date": "2018-12-31",
    "hourly": "temperature_2m",
}
timeout = 30

def get_airport_weather(start_date = '2016-01-01', end_date = '2018-12-31'):
    all_weather = []
    for airport_name, (lat, lon) in AIRPORTS.items():
        params= {
            'latitude': lat, 'longitude': lon, 'start_date': start_date, 'end_date': end_date,
            'daily': ['temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'snowfall_sum',
                'windspeed_10m_max', 'weathercode', ],
            'time_zone': 'auto'}
        response = requests.get(url, params=params, timeout=timeout)
        data = response.json()
        daily = data["daily"]

        airport_df = pd.DataFrame({
        'airport_name': airport_name,
        'date': daily['time'],
        'temp_max_c': daily['temperature_2m_max'],
        'temp_min_c': daily['temperature_2m_min'],
        'precipitation_mm': daily['precipitation_sum'],
        'wind_speed_mph': daily['windspeed_10m_max'],
        'snow_mm': daily['snowfall_sum'],
        'weather_code': daily['weathercode'],
        
    })
        all_weather.append(airport_df)
    
    return pd.concat(all_weather, ignore_index=True)

In [10]:
airport_data  = get_airport_weather()
airport_weather = pd.DataFrame(airport_data)

#### Looking at the Structure of CSV

In [11]:
takeoffs2016[takeoffs2016["DEP_TIME"].isna()][["FL_DATE", "DEP_DELAY", "DEP_TIME"]]


,FL_DATE,DEP_DELAY,DEP_TIME
699,2016-01-01,NaN,NaN
710,2016-01-01,NaN,NaN
713,2016-01-01,NaN,NaN
1231,2016-01-01,NaN,NaN
1242,2016-01-01,NaN,NaN
...,...,...,...
5611879,2016-12-31,NaN,NaN
5614160,2016-12-31,NaN,NaN
5614414,2016-12-31,NaN,NaN
5616755,2016-12-31,NaN,NaN


In [12]:
print(takeoffs2016.dtypes)

FL_DATE                    str
OP_CARRIER                 str
OP_CARRIER_FL_NUM        int64
ORIGIN                     str
DEST                       str
CRS_DEP_TIME             int64
DEP_TIME               float64
DEP_DELAY              float64
TAXI_OUT               float64
WHEELS_OFF             float64
WHEELS_ON              float64
TAXI_IN                float64
CRS_ARR_TIME             int64
ARR_TIME               float64
ARR_DELAY              float64
CANCELLED              float64
CANCELLATION_CODE          str
DIVERTED               float64
CRS_ELAPSED_TIME       float64
ACTUAL_ELAPSED_TIME    float64
AIR_TIME               float64
DISTANCE               float64
CARRIER_DELAY          float64
WEATHER_DELAY          float64
NAS_DELAY              float64
SECURITY_DELAY         float64
LATE_AIRCRAFT_DELAY    float64
Unnamed: 27            float64
dtype: object


In [13]:
print(takeoffs2017.dtypes)

FL_DATE                    str
OP_CARRIER                 str
OP_CARRIER_FL_NUM        int64
ORIGIN                     str
DEST                       str
CRS_DEP_TIME             int64
DEP_TIME               float64
DEP_DELAY              float64
TAXI_OUT               float64
WHEELS_OFF             float64
WHEELS_ON              float64
TAXI_IN                float64
CRS_ARR_TIME             int64
ARR_TIME               float64
ARR_DELAY              float64
CANCELLED              float64
CANCELLATION_CODE          str
DIVERTED               float64
CRS_ELAPSED_TIME       float64
ACTUAL_ELAPSED_TIME    float64
AIR_TIME               float64
DISTANCE               float64
CARRIER_DELAY          float64
WEATHER_DELAY          float64
NAS_DELAY              float64
SECURITY_DELAY         float64
LATE_AIRCRAFT_DELAY    float64
Unnamed: 27            float64
dtype: object


In [14]:
print(takeoffs2018.dtypes)

FL_DATE                    str
OP_CARRIER                 str
OP_CARRIER_FL_NUM        int64
ORIGIN                     str
DEST                       str
CRS_DEP_TIME             int64
DEP_TIME               float64
DEP_DELAY              float64
TAXI_OUT               float64
WHEELS_OFF             float64
WHEELS_ON              float64
TAXI_IN                float64
CRS_ARR_TIME             int64
ARR_TIME               float64
ARR_DELAY              float64
CANCELLED              float64
CANCELLATION_CODE          str
DIVERTED               float64
CRS_ELAPSED_TIME       float64
ACTUAL_ELAPSED_TIME    float64
AIR_TIME               float64
DISTANCE               float64
CARRIER_DELAY          float64
WEATHER_DELAY          float64
NAS_DELAY              float64
SECURITY_DELAY         float64
LATE_AIRCRAFT_DELAY    float64
Unnamed: 27            float64
dtype: object


### Cleaning data

##### Takeoff changes

In [15]:
takeoffs2016['FL_DATE'] = pd.to_datetime(takeoffs2016['FL_DATE'], format = 'mixed')

In [16]:
takeoffs2017['FL_DATE'] = pd.to_datetime(takeoffs2017['FL_DATE'], format = 'mixed')

In [17]:
takeoffs2018['FL_DATE'] = pd.to_datetime(takeoffs2018['FL_DATE'], format = 'mixed')

In [18]:
print(takeoffs2016['FL_DATE'].head())
print(takeoffs2017['FL_DATE'].head())
print(takeoffs2018['FL_DATE'].head())

0   2016-01-01
1   2016-01-01
2   2016-01-01
3   2016-01-01
4   2016-01-01
Name: FL_DATE, dtype: datetime64[us]
0   2017-01-01
1   2017-01-01
2   2017-01-01
3   2017-01-01
4   2017-01-01
Name: FL_DATE, dtype: datetime64[us]
0   2018-01-01
1   2018-01-01
2   2018-01-01
3   2018-01-01
4   2018-01-01
Name: FL_DATE, dtype: datetime64[us]


##### Weather changing

In [22]:
airport_weather['date'] = pd.to_datetime(airport_weather['date'])

def wmo_condition(code):
    if code == 0:
        return "Clear"
    elif code in [1,2,3]:
        return "Cloudy"
    elif 51 <= code <= 57:
        return "Drizzle"
    elif 61 <= code <= 67:
        return "Rain"
    elif 71 <= code <= 75:
        return "Snow"
    elif 80 <= code <= 82:
        return "Showers"
    elif 95 <= code <= 99:
        return "Thunderstorm"
    else:
        return "Other"

    

In [ ]:
airport_weather.head()
airport_weather.describe()

,airport_name,date,temp_max_c,temp_min_c,precipitation_mm,wind_speed_mph,snow_mm,weather_code
0,Atlanta ATL,2016-01-01,-5.6,-22.0,0.0,10.5,0.00,3
1,Atlanta ATL,2016-01-02,-8.8,-22.5,0.0,8.2,0.00,3
2,Atlanta ATL,2016-01-03,-9.6,-24.2,0.0,9.0,0.00,3
3,Atlanta ATL,2016-01-04,-4.4,-25.2,0.0,9.2,0.00,3
4,Atlanta ATL,2016-01-05,-8.1,-27.5,1.3,9.9,1.05,71
